In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00


In [ ]:
!pip install -q "transformers==4.48.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from datasets import load_dataset
ds = load_dataset("lmms-lab/HallusionBench")
print(ds)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 951
    })
    non_image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 178
    })
})


In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "microsoft/Phi-3.5-vision-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    _attn_implementation="eager",   # avoids needing flash_attn installed
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    num_crops=4,   # caps image tiling to control memory — same role as max_pixels/max_num in the earlier models
)
print("Loaded.")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/auto/image_processing_auto.py:590: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


Loaded.


In [ ]:
def parse_yes_no(text):
    a = text.strip().lower()
    if a.startswith("yes"): return "1"
    if a.startswith("no"): return "0"
    return "unclear"

def ask(question, image=None):
    if image is not None:
        prompt = f"<|user|>\n<|image_1|>\n{question} Please answer with only Yes or No.<|end|>\n<|assistant|>\n"
        inputs = processor(prompt, [image.convert("RGB")], return_tensors="pt").to(model.device)
    else:
        prompt = f"<|user|>\n{question} Please answer with only Yes or No.<|end|>\n<|assistant|>\n"
        inputs = processor(prompt, return_tensors="pt").to(model.device)

    generate_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        eos_token_id=processor.tokenizer.eos_token_id,
        do_sample=False,
    )
    generate_ids = generate_ids[:, inputs["input_ids"].shape[1]:]
    response = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

    del inputs, generate_ids
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    return response

In [ ]:
ex = ds["image"][0]
r = ask(ex["question"], ex["image"])
print("IMAGE Q GT:", ex["gt_answer"], "| Response:", repr(r), "| Parsed:", parse_yes_no(r))

ex2 = ds["non_image"][0]
r2 = ask(ex2["question"], None)
print("TEXT Q GT:", ex2["gt_answer"], "| Response:", repr(r2), "| Parsed:", parse_yes_no(r2))

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


IMAGE Q GT: 0 | Response: 'Yes' | Parsed: 1
TEXT Q GT: 0 | Response: 'Yes' | Parsed: 1


In [ ]:
import time
start = time.time()
for ex in ds["image"].select(range(20)):
    ask(ex["question"], ex["image"])
elapsed = time.time() - start
print(f"20 questions: {elapsed:.1f}s | Estimated full 1129: {elapsed*1129/20/60:.1f} min")

20 questions: 27.8s | Estimated full 1129: 26.2 min


In [ ]:
import json, os, gc

RESULTS_PATH = "/content/phi35vision_results.json"

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
else:
    results = []

done = {(r["question_id"], r["split"]) for r in results}

def save():
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)

def run_split(split_name):
    n = len(ds[split_name])
    for i, ex in enumerate(ds[split_name]):
        if (ex["question_id"], split_name) in done:
            continue

        img = ex["image"] if split_name == "image" else None

        try:
            resp = ask(ex["question"], img)
        except Exception as e:
            gc.collect(); torch.cuda.empty_cache()
            resp = f"ERROR: {type(e).__name__}: {str(e)[:100]}"

        results.append({
            "question_id": ex["question_id"],
            "split": split_name,
            "category": ex["category"],
            "subcategory": ex["subcategory"],
            "figure_id": ex["figure_id"],
            "set_id": ex["set_id"],
            "question": ex["question"],
            "gt_answer": ex["gt_answer"],
            "raw_response": resp,
            "model_prediction": parse_yes_no(resp),
        })

        if (i + 1) % 50 == 0:
            save()
            print(f"[{split_name}] {i+1}/{n}")
    save()

run_split("image")
run_split("non_image")

print("Done:", len(results))

[image] 50/951
[image] 100/951
[image] 150/951
[image] 200/951
[image] 250/951
[image] 300/951
[image] 350/951
[image] 400/951
[image] 450/951
[image] 500/951


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[image] 550/951
[image] 600/951
[image] 650/951
[image] 700/951
[image] 750/951
[image] 800/951
[image] 850/951
[image] 900/951
[image] 950/951
[non_image] 50/178
[non_image] 100/178
[non_image] 150/178
Done: 1129


In [ ]:
from collections import defaultdict

def compute_metrics(rs):
    total = len(rs)
    if total == 0: return None
    ok = lambda r: r["model_prediction"] == str(r["gt_answer"])
    aAcc = sum(ok(r) for r in rs) / total * 100
    sets = defaultdict(list)
    for r in rs: sets[(r["category"], r["subcategory"], r["set_id"])].append(r)
    qAcc = sum(all(ok(m) for m in v) for v in sets.values()) / len(sets) * 100
    figs = defaultdict(list)
    for r in rs: figs[(r["category"], r["subcategory"], r["figure_id"])].append(r)
    fAcc = sum(all(ok(m) for m in v) for v in figs.values()) / len(figs) * 100
    return {"aAcc": round(aAcc,2), "qAcc": round(qAcc,2), "fAcc": round(fAcc,2),
            "n_questions": total, "n_sets": len(sets), "n_figures": len(figs)}

print("=== Overall ===")
print(compute_metrics(results))

by_cat = defaultdict(list)
for r in results: by_cat[r["category"]].append(r)
print("\n=== By Category ===")
for k, v in by_cat.items(): print(k, compute_metrics(v))

by_sub = defaultdict(list)
for r in results: by_sub[(r["category"], r["subcategory"])].append(r)
print("\n=== By Category + Subcategory ===")
for k, v in sorted(by_sub.items()): print(k, compute_metrics(v))

print("\nUnclear/error:", sum(r["model_prediction"]=="unclear" or str(r["raw_response"]).startswith("ERROR") for r in results), "/", len(results))

=== Overall ===
{'aAcc': 61.2, 'qAcc': 9.58, 'fAcc': 3.85, 'n_questions': 1129, 'n_sets': 167, 'n_figures': 26}

=== By Category ===
VS {'aAcc': 63.94, 'qAcc': 1.75, 'fAcc': 0.0, 'n_questions': 538, 'n_sets': 57, 'n_figures': 13}
VD {'aAcc': 58.71, 'qAcc': 13.64, 'fAcc': 7.69, 'n_questions': 591, 'n_sets': 110, 'n_figures': 13}

=== By Category + Subcategory ===
('VD', 'figure') {'aAcc': 67.5, 'qAcc': 20.0, 'fAcc': 33.33, 'n_questions': 80, 'n_sets': 20, 'n_figures': 3}
('VD', 'illusion') {'aAcc': 59.03, 'qAcc': 9.68, 'fAcc': 0.0, 'n_questions': 144, 'n_sets': 31, 'n_figures': 2}
('VD', 'math') {'aAcc': 56.48, 'qAcc': 5.56, 'fAcc': 0.0, 'n_questions': 108, 'n_sets': 18, 'n_figures': 2}
('VD', 'ocr') {'aAcc': 73.03, 'qAcc': 33.33, 'fAcc': 0.0, 'n_questions': 89, 'n_sets': 21, 'n_figures': 3}
('VD', 'video') {'aAcc': 48.24, 'qAcc': 0.0, 'fAcc': 0.0, 'n_questions': 170, 'n_sets': 20, 'n_figures': 3}
('VS', 'chart') {'aAcc': 65.53, 'qAcc': 4.55, 'fAcc': 0.0, 'n_questions': 206, 'n_sets': 2

In [ ]:
for r in results:
    if r["model_prediction"] == "unclear" or str(r["raw_response"]).startswith("ERROR"):
        print(r["question_id"], "|", repr(r["raw_response"]))

0 | 'Syria'
0 | 'Niger'
0 | 'Germany'
0 | 'France'
0 | 'Germany'
0 | 'Florida'
0 | 'Florida'
0 | 'South Carolina'


In [ ]:
metrics_summary = {
    "model": "phi-3.5-vision-instruct",
    "overall": compute_metrics(results),
    "by_category": {c: compute_metrics(v) for c, v in by_cat.items()},
    "by_category_subcategory": {f"{k[0]}_{k[1]}": compute_metrics(v) for k, v in by_sub.items()},
}
with open("/content/phi35vision_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

from google.colab import files
files.download("/content/phi35vision_results.json")
files.download("/content/phi35vision_metrics.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>